# Train with PPO

Trains a policy on `UniWheelEnv` using the shared `rl/ppo_trainer.py` (same trainer
the reference repo uses, unmodified - it's environment-agnostic).

**Already smoke-tested**: a 5-iteration run (4 envs x 2048 steps x 5 = 40,960 steps)
completed end-to-end and mean episodic return climbed 52 -> 66 -> 68 -> 69 over those
5 iterations, with checkpointing and best-model saving both confirmed working. That's
nowhere near enough to expect a balanced policy - it just confirms the env/trainer/
reward wiring is correct before you commit to a real run. `STEPS_PER_ENV` below is set
for an actual training run; drop it if you want to re-confirm the wiring first.

**GPU note**: measured it directly rather than assuming - on this model (tiny 64-unit
MLP, cheap 4-body MuJoCo sim), `cuda=True` was actually *slower* than CPU (14.9s vs
9.8s for the same 5-iteration test) - PCIe transfer overhead for such small tensors
outweighs anything the GPU saves. The real lever here is `NUM_ENVS`: going from 4 to 12
(matching this machine's core count) measured ~1.5x more steps/sec with zero added
complexity, so that's what's set below. (True multiprocess parallelism via
`AsyncVectorEnv` was also tried - only ~10% faster than plain `SyncVectorEnv` at the
same env count, and it needs a small shim since the trainer expects `envs.envs[0]` for
rendering - not worth the added complexity for that little gain, so not used here.)


In [1]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch


In [2]:
# Add the folder containing the rl/ package to the path
sys.path.append("..")

# Import the custom environment and PPO training module
from uniwheel_env import UniWheelEnv
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv


In [3]:
# Settings
MJCF_PATH = Path("../../mechanical/urdf/uniwheel/uniwheel.xml")
SEED = 42
NUM_ENVS = 12                 # Number of parallel environments (matches this machine's CPU
                               # core count - see the GPU/parallelism note above for why this,
                               # not cuda, is the actual speed lever here). Only the first is rendered.
STEPS_PER_ENV = 1_000_000     # Number of simulation steps to perform per environment


In [4]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "uniwheel-ppo",     # Name of the experiment
    env_id = "UniWheel-v0",        # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    cuda = False,                  # Measured slower than CPU for this tiny network + cheap
                                    # sim (see markdown note above) - not a typo
    num_envs = NUM_ENVS,           # Number of parallel environments
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.005s = ~10 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic weights change per update
    checkpoint_interval = 20,      # Save a checkpoint every N iterations
    save_model = True,             # Save the final model too
)


In [5]:
def make_uniwheel_env(render):
    """Function to create an environment for the uni wheel robot"""
    env = UniWheelEnv(
        mjcf_path=MJCF_PATH,
        render_mode="human" if render else None,
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)


In [6]:
# Build list of environment factory functions
env_factories = []
for i in range(NUM_ENVS):
    env_factories.append(lambda render=(i==0): make_uniwheel_env(render))

# Build vector of environments
envs = gym.vector.SyncVectorEnv(env_factories)


In [7]:
# Choo choo train!
result = train(ppo_config, envs=envs)


Run name: UniWheel-v0__uniwheel-ppo__42__1788678088
TensorBoard: http://localhost:6006/#scalars&regexFilter=uniwheel-ppo


Checkpoint saved to runs/UniWheel-v0__uniwheel-ppo__42__1788678088/checkpoint_iter0020.cleanrl_model
New best model saved (mean_return=430.20) to runs/UniWheel-v0__uniwheel-ppo__42__1788678088/best_model.cleanrl_model


Checkpoint saved to runs/UniWheel-v0__uniwheel-ppo__42__1788678088/checkpoint_iter0040.cleanrl_model
New best model saved (mean_return=6187.09) to runs/UniWheel-v0__uniwheel-ppo__42__1788678088/best_model.cleanrl_model


Checkpoint saved to runs/UniWheel-v0__uniwheel-ppo__42__1788678088/checkpoint_iter0060.cleanrl_model
New best model saved (mean_return=9092.92) to runs/UniWheel-v0__uniwheel-ppo__42__1788678088/best_model.cleanrl_model


Checkpoint saved to runs/UniWheel-v0__uniwheel-ppo__42__1788678088/checkpoint_iter0080.cleanrl_model


...

Checkpoint saved to runs/UniWheel-v0__uniwheel-ppo__42__1788678088/checkpoint_iter0420.cleanrl_model


Checkpoint saved to runs/UniWheel-v0__uniwheel-ppo__42__1788678088/checkpoint_iter0440.cleanrl_model


Checkpoint saved to runs/UniWheel-v0__uniwheel-ppo__42__1788678088/checkpoint_iter0460.cleanrl_model


Checkpoint saved to runs/UniWheel-v0__uniwheel-ppo__42__1788678088/checkpoint_iter0480.cleanrl_model


Final model saved to runs/UniWheel-v0__uniwheel-ppo__42__1788678088/uniwheel-ppo_final.cleanrl_model
Best model mean return: 9970.52, saved to runs/UniWheel-v0__uniwheel-ppo__42__1788678088/best_model.cleanrl_model


In [8]:
# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")


Best model: runs/UniWheel-v0__uniwheel-ppo__42__1788678088/best_model.cleanrl_model
Final model: runs/UniWheel-v0__uniwheel-ppo__42__1788678088/uniwheel-ppo_final.cleanrl_model
Best mean return: 9970.52
Loaded best model (mean_return=9970.52)


In [9]:
# Switch to real-time timestep for evaluation
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent,
    eval_episodes=3,
    config=eval_config,
    envs=envs)

print(f"Mean return: {np.nanmean(returns):.2f}")


Mean return: 2040.55


In [10]:
# Close the environments
envs.close()


In [11]:
# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)


Exported charts_metrics.csv (5 metrics, 4879 steps)
Exported losses_metrics.csv (7 metrics, 488 steps)
